# 13 — Symmetric Mode Pairs

**Version:** `13_symmetric_mode_pairs_v1`

Notebook 07 established:

\[
f_n = f_0 + n\Delta f
\]

Notebook 13 asks:

> **Given many frequency modes, which modes become naturally paired?**

The symmetry is:

\[
2\omega_0 = \omega_{+n} + \omega_{-n}
\]

or:

\[
\omega_p + \omega_p \rightarrow \omega_{+n} + \omega_{-n}
\]

This notebook treats symmetric frequency modes as a **pair graph**.

In [ ]:
from pathlib import Path
import json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

VERSION = "13_symmetric_mode_pairs_v1"
print("running:", VERSION)

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    ROOT = cwd.parent
elif (cwd / "notebooks").exists():
    ROOT = cwd
else:
    ROOT = cwd

FIGURES_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
CSV_DIR = RESULTS_DIR / "csv"
JSON_DIR = RESULTS_DIR / "json"

for path in [FIGURES_DIR, CSV_DIR, JSON_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)

## 1. Build symmetric pairs

Symmetric mode pairs have the form:

\[
(-n,+n)
\]

The idealized conservation check is:

\[
(-n) + (+n) = 0
\]

In [ ]:
n_pairs = 8
pair_indices = np.arange(1, n_pairs + 1)

pairs = pd.DataFrame({
    "pair_index": pair_indices,
    "left_mode": -pair_indices,
    "right_mode": pair_indices,
})
pairs["relative_sum"] = pairs["left_mode"] + pairs["right_mode"]
pairs["conservation_check"] = np.where(pairs["relative_sum"] == 0, "symmetric", "not symmetric")

pairs_path = CSV_DIR / "13_symmetric_pairs.csv"
pairs.to_csv(pairs_path, index=False)
pairs

## 2. Symmetric pair structure

The comb becomes a structured set of symmetric mode pairs.

In [ ]:
mode_indices = np.arange(-n_pairs, n_pairs + 1)

fig, ax = plt.subplots(figsize=(11, 4))
ax.vlines(mode_indices, 0, 1, linewidth=1)
ax.scatter(mode_indices, np.ones_like(mode_indices), s=32)
ax.axvline(0, linestyle="--", alpha=0.6)
ax.text(0, 1.12, "pump\nf₀", ha="center", va="bottom")

for idx, n in enumerate(pair_indices):
    y = 0.82 - idx * 0.055
    ax.plot([-n, n], [y, y], linewidth=1.4)
    ax.scatter([-n, n], [y, y], s=14)

ax.set_xticks([-8, -4, -1, 0, 1, 4, 8])
ax.set_xticklabels(["f₀−8Δf", "f₀−4Δf", "f₀−Δf", "f₀", "f₀+Δf", "f₀+4Δf", "f₀+8Δf"])
ax.set_title("Symmetric Frequency-Mode Pair Structure")
ax.set_xlabel("Frequency mode")
ax.set_yticks([])
ax.set_ylim(0.25, 1.25)
ax.set_xlim(mode_indices.min() - 1, mode_indices.max() + 1)

fig.tight_layout()
pair_structure_path = FIGURES_DIR / "13_symmetric_pair_structure.png"
fig.savefig(pair_structure_path, dpi=200)
plt.show()

print("saved:", pair_structure_path)

## 3. Pair adjacency matrix

Each frequency mode is a node.

Each symmetric pair \((-n,+n)\) is an edge.

In [ ]:
nodes = list(range(-n_pairs, 0)) + list(range(1, n_pairs + 1))
node_to_idx = {node: idx for idx, node in enumerate(nodes)}
adjacency = np.zeros((len(nodes), len(nodes)), dtype=int)

for _, row in pairs.iterrows():
    left = int(row["left_mode"])
    right = int(row["right_mode"])
    i = node_to_idx[left]
    j = node_to_idx[right]
    adjacency[i, j] = 1
    adjacency[j, i] = 1

adjacency_df = pd.DataFrame(adjacency, index=nodes, columns=nodes)
adjacency_path = CSV_DIR / "13_pair_adjacency_matrix.csv"
adjacency_df.to_csv(adjacency_path)
adjacency_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(adjacency, interpolation="nearest")
ax.set_title("Adjacency Matrix for Symmetric Mode Pairs")
ax.set_xlabel("Mode index")
ax.set_ylabel("Mode index")
ax.set_xticks(np.arange(len(nodes)))
ax.set_yticks(np.arange(len(nodes)))
ax.set_xticklabels(nodes, rotation=90)
ax.set_yticklabels(nodes)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="edge")

fig.tight_layout()
adjacency_figure_path = FIGURES_DIR / "13_pair_adjacency_matrix.png"
fig.savefig(adjacency_figure_path, dpi=200)
plt.show()

print("saved:", adjacency_figure_path)

## 4. Pair graph

The graph view bridges toward multipartite entanglement networks.

In [ ]:
G = nx.Graph()
for _, row in pairs.iterrows():
    left = int(row["left_mode"])
    right = int(row["right_mode"])
    G.add_edge(left, right, pair_index=int(row["pair_index"]))

graph_summary = {
    "nodes": G.number_of_nodes(),
    "edges": G.number_of_edges(),
    "connected_components": nx.number_connected_components(G),
    "component_sizes": [len(c) for c in nx.connected_components(G)],
}

graph_summary_path = JSON_DIR / "13_pair_graph_metrics.json"
graph_summary_path.write_text(json.dumps(graph_summary, indent=2))
graph_summary

In [ ]:
pos = {}
for idx, n in enumerate(range(1, n_pairs + 1)):
    y = n_pairs - idx
    pos[-n] = (-1, y)
    pos[n] = (1, y)

fig, ax = plt.subplots(figsize=(7, 7))
nx.draw_networkx_nodes(G, pos, node_size=700, ax=ax)
nx.draw_networkx_edges(G, pos, width=2, ax=ax)
nx.draw_networkx_labels(G, pos, labels={node: str(node) for node in G.nodes()}, font_size=10, ax=ax)

ax.set_title("Pair Graph: Symmetric Frequency Modes")
ax.axis("off")

fig.tight_layout()
graph_path = FIGURES_DIR / "13_pair_graph.png"
fig.savefig(graph_path, dpi=200)
plt.show()

print("saved:", graph_path)

## 5. Summary

Frequency multiplexing supplies many modes.

Kerr symmetry selects structured pairs:

\[
(-1,+1),\ (-2,+2),\ \ldots,\ (-n,+n)
\]

This converts a frequency comb into a pair graph.

In [ ]:
summary = {
    "notebook": "13_symmetric_mode_pairs",
    "version": VERSION,
    "question": "Given many frequency modes, which modes become naturally paired?",
    "core_relation": "2 omega_0 = omega_{+n} + omega_{-n}",
    "four_wave_mixing_form": "omega_p + omega_p -> omega_{+n} + omega_{-n}",
    "n_pairs": int(n_pairs),
    "outputs": [
        "figures/13_symmetric_pair_structure.png",
        "figures/13_pair_adjacency_matrix.png",
        "figures/13_pair_graph.png",
        "results/csv/13_symmetric_pairs.csv",
        "results/csv/13_pair_adjacency_matrix.csv",
        "results/json/13_pair_graph_metrics.json",
        "results/json/13_symmetric_mode_pairs_summary.json",
        "results/13_symmetric_mode_pairs_outputs.zip"
    ],
}
summary_path = JSON_DIR / "13_symmetric_mode_pairs_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

## 6. Download outputs

Run this cell to package all Notebook 13 outputs.

In Google Colab, it starts a browser download.

In local Jupyter, it prints the zip path.

In [ ]:
zip_path = RESULTS_DIR / "13_symmetric_mode_pairs_outputs.zip"

files_to_zip = [
    FIGURES_DIR / "13_symmetric_pair_structure.png",
    FIGURES_DIR / "13_pair_adjacency_matrix.png",
    FIGURES_DIR / "13_pair_graph.png",
    CSV_DIR / "13_symmetric_pairs.csv",
    CSV_DIR / "13_pair_adjacency_matrix.csv",
    JSON_DIR / "13_pair_graph_metrics.json",
    JSON_DIR / "13_symmetric_mode_pairs_summary.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print("download package ready:", zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Local Jupyter: download manually from")
    print(zip_path)

In [ ]:
outputs = [
    FIGURES_DIR / "13_symmetric_pair_structure.png",
    FIGURES_DIR / "13_pair_adjacency_matrix.png",
    FIGURES_DIR / "13_pair_graph.png",
    CSV_DIR / "13_symmetric_pairs.csv",
    CSV_DIR / "13_pair_adjacency_matrix.csv",
    JSON_DIR / "13_pair_graph_metrics.json",
    JSON_DIR / "13_symmetric_mode_pairs_summary.json",
    RESULTS_DIR / "13_symmetric_mode_pairs_outputs.zip",
]

for output in outputs:
    print("exists:", output.exists(), "→", output.relative_to(ROOT) if output.exists() else output)

## Takeaway

Notebook 07 established the comb as many addressable frequency modes.

Notebook 13 adds the first quantum-optical structure:

\[
(-n,+n)
\]

Those symmetric pairs form a graph.

Notebook 23 can now ask how pair graphs extend into multipartite entanglement networks.